In [2]:
context="The document does not specify who was the president of the Indian National Congress when India became free. However, it mentions that Jivatram Bhagwandas Kripalani was the president of the Indian National Congress during the transfer of power in 1947.There is no context information provided for this query. The Indian National Congress, led by Mahatma Gandhi, played a key role in India's independence from the United Kingdom in 1947. Jivatram Bhagwandas Kripalani was the president of the Indian National Congress when India became free in 1947, marking the culmination of the party's efforts to achieve independence."

question='Who was the president of indian national congress when india became free?'

follow=['Who was the president of indian national congress when india became free due to the Indian Independence Act 1947 being passed?', 'Who was the president of indian national congress when india became free due to the Constitution of India taking effect?']
short=[['J. B. Kripalani', 'Jivatram Bhagwandas Kripalani', 'Acharya Kripalani'], ['Purushottam Das Tandon']]

In [3]:
import argparse
import collections
import json
import re
import string


def normalize_answer(s):
  """Lower text and remove punctuation, articles and extra whitespace."""

  def remove_articles(text):
    return re.sub(r'\b(a|an|the)\b', ' ', text)

  def white_space_fix(text):
    return ' '.join(text.split())

  def remove_punc(text):
    exclude = set(string.punctuation)
    return ''.join(ch for ch in text if ch not in exclude)

  def lower(text):
    return text.lower()

  return white_space_fix(remove_articles(remove_punc(lower(s))))

def _get_tokens(s):
  """Split the string into tokens.

  Args:
    s: string to be split

  Returns:
    list of tokens
  """

  if not s:
    return []
  return normalize_answer(s).split()

def _compute_f1(a_gold, a_pred):
  """Compute F1 score between two strings.

  Args:
    a_gold: string one
    a_pred: string two

  Returns:
        f1 score
  """
  a_gold_str = " ".join(a_gold)
  a_pred_str = " ".join(a_pred)
  
  gold_toks = _get_tokens(a_gold_str)
  pred_toks = _get_tokens(a_pred_str)

  common = collections.Counter(gold_toks) & collections.Counter(pred_toks)
  num_same = sum(common.values())

  if len(gold_toks) == 0 or len(pred_toks) == 0:
    # If either is no-answer, then F1 is 1 if they agree, 0 otherwise
    return int(gold_toks == pred_toks)

  if num_same == 0:
    return 0

  precision = 1.0 * num_same / len(pred_toks)
  recall = 1.0 * num_same / len(gold_toks)
  f1 = (2 * precision * recall) / (precision + recall)

  return f1

In [4]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_name = "deepset/roberta-base-squad2"

prediction=[]

nlp = pipeline('question-answering', model=model_name, tokenizer=model_name, device=device)
for i in range(len(follow)):
    QA_input = {
        'question': follow[i],
        'context': context
    }
    res = nlp(QA_input)
    prediction.append(res['answer'])
    print(res)
    print(short[i])

/home/explorer/anaconda3/envs/djk_sf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/explorer/anaconda3/envs/djk_sf/lib/python3.11/site-packages/transformers/pipelines/question_answering.py:391: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


{'score': 0.2580374777317047, 'start': 134, 'end': 163, 'answer': 'Jivatram Bhagwandas Kripalani'}
['J. B. Kripalani', 'Jivatram Bhagwandas Kripalani', 'Acharya Kripalani']
{'score': 0.42203184962272644, 'start': 134, 'end': 163, 'answer': 'Jivatram Bhagwandas Kripalani'}
['Purushottam Das Tandon']


In [5]:
answers=short

In [6]:
answers

[['J. B. Kripalani', 'Jivatram Bhagwandas Kripalani', 'Acharya Kripalani'],
 ['Purushottam Das Tandon']]

In [18]:
prediction

['Jivatram Bhagwandas Kripalani', 'Jivatram Bhagwandas Kripalani']

In [8]:
a=['J. B. Kripalani', 'Jivatram Bhagwandas Kripalani', 'Acharya Kripalani']
p='Jivatram Bhagwandas Kripalani'

loc_f1=list(_compute_f1(a, p) for a in answers for p in prediction)

In [19]:
loc_f1

[0.14285714285714288, 0.14285714285714288, 0, 0]

In [9]:
res=_compute_f1('Jivatram Bhagwandas Kripalani','Jivatram Bhagwandas Kripalani')

In [10]:
res

1.0